# PDF/Text Extraction Analysis — Member 2

**Goal:** Compare text-extraction quality between the Kaggle PDF dataset and the college-material PDF dataset using the two supplied PDFs: `kaggle.pdf` and `clgdataset.pdf`.

This notebook measures pages, words, characters, extractable text, and page-level text density. It also creates a manual quality-review sheet for missing/broken text, reading order, headers/footers, symbols, equations, tables, and repeated content.

**Important:** The automatic checks are screening indicators, not proof that extraction is perfect. Inspect the extracted `.txt` files alongside the original PDFs before making conclusions.

## 1. Install the PDF extraction library

Run this cell in Google Colab. PyMuPDF is used to extract selectable text from PDFs.

In [18]:
%pip -q install pymupdf


## 2. Upload the two PDFs

Upload the files named `kaggle.pdf` and `clgdataset.pdf`. If Colab gives them different filenames, the next cell lets you assign their source labels manually.

In [19]:
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


Saving kaggle.pdf to kaggle (2).pdf
Saving clgdataset.pdf to clgdataset (3).pdf
Uploaded files: ['kaggle (2).pdf', 'clgdataset (3).pdf']


## 3. Map filenames to sources

The default mapping matches the filenames you provided. If necessary, edit the filenames below to match the names shown after upload. Source labels are used in the final comparison.

In [21]:
import os

print("Files in Colab:")
print(os.listdir('/content'))

pdfs = [
    f for f in os.listdir('/content')
    if f.lower().endswith('.pdf')
]

kaggle_file = next(
    (f for f in pdfs if f.lower().startswith('kaggle')),
    None
)

college_file = next(
    (f for f in pdfs if f.lower().startswith('clgdataset')),
    None
)

if kaggle_file is None or college_file is None:
    print("One or both PDFs are missing. Please upload them again.")
else:
    SOURCE_FILES = {
        'Kaggle': kaggle_file,
        'College': college_file
    }

    print("\nSource mapping ready:")
    for source, filename in SOURCE_FILES.items():
        print(f"{source}: {filename}")

Files in Colab:
['.config', 'pdf_extraction_analysis_results.zip', 'clgdataset (3).pdf', 'clgdataset (1).pdf', 'clgdataset.pdf', 'kaggle.pdf', 'clgdataset (2).pdf', 'kaggle (2).pdf', 'kaggle (1).pdf', 'extraction_results', 'sample_data']

Source mapping ready:
Kaggle: kaggle.pdf
College: clgdataset (3).pdf


In [22]:
print("Files currently in uploaded:")
for name in uploaded.keys():
    print(repr(name))

Files currently in uploaded:
'kaggle (2).pdf'
'clgdataset (3).pdf'


## 4. Extract text and calculate document/page statistics

Definitions used here:
- **Words:** tokens matching a basic Unicode word pattern.
- **Characters:** characters in extracted, cleaned text, including spaces and punctuation.
- **Empty pages:** pages where no text was extracted.
- **Low-text pages:** pages with fewer than 50 extracted characters; these require inspection and may be image-only pages, diagrams, title pages, or genuinely sparse pages.
- **Extractable-text percentage:** percentage of pages with any extracted text. This is not the percentage of all original text recovered.

In [23]:
import fitz
import re
import os
import pandas as pd
from collections import Counter
from IPython.display import display

os.makedirs('extraction_results/extracted_text', exist_ok=True)
document_rows = []
page_rows = []
repeat_rows = []
extracted_texts = {}

def clean_text(text):
    text = text.replace('\x00', '')
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

for source, filename in SOURCE_FILES.items():
    doc = fitz.open(filename)
    page_texts = []
    for page_num, page in enumerate(doc, start=1):
        raw_text = page.get_text('text', sort=True)
        text = clean_text(raw_text)
        words = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
        chars = len(text)
        page_texts.append(text)
        page_rows.append({
            'source': source,
            'filename': filename,
            'page': page_num,
            'words': len(words),
            'characters': chars,
            'empty_page': chars == 0,
            'low_text_page_under_50_chars': chars < 50,
        })

    all_text = '\n\n'.join(
        f'===== PAGE {i + 1} =====\n{text}'
        for i, text in enumerate(page_texts)
    )
    extracted_texts[source] = all_text
    safe_source = re.sub(r'[^A-Za-z0-9_-]+', '_', source)
    text_path = f'extraction_results/extracted_text/{safe_source}_extracted.txt'
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(all_text)

    total_words = len(re.findall(r'\b\w+\b', all_text, flags=re.UNICODE))
    total_chars = len(all_text)
    page_count = len(doc)
    empty_pages = sum(not text for text in page_texts)
    low_pages = sum(len(text) < 50 for text in page_texts)
    file_size = os.path.getsize(filename)

    lines = [re.sub(r'\s+', ' ', line).strip()
             for text in page_texts for line in text.splitlines()
             if len(line.strip()) > 2]
    line_counts = Counter(lines)
    repeated_lines = [(line, count) for line, count in line_counts.items() if count >= 3]
    for line, count in repeated_lines:
        repeat_rows.append({'source': source, 'filename': filename,
                            'line': line, 'occurrences': count})

    document_rows.append({
        'source': source,
        'filename': filename,
        'file_size_KB': round(file_size / 1024, 2),
        'pages': page_count,
        'total_words': total_words,
        'total_characters': total_chars,
        'average_words_per_page': round(total_words / page_count, 2) if page_count else 0,
        'average_characters_per_page': round(total_chars / page_count, 2) if page_count else 0,
        'empty_pages': empty_pages,
        'low_text_pages': low_pages,
        'extractable_pages_percent': round(100 * (page_count - empty_pages) / page_count, 2) if page_count else 0,
        'repeated_lines_3plus': len(repeated_lines),
    })
    doc.close()

document_stats = pd.DataFrame(document_rows)
page_stats = pd.DataFrame(page_rows)
repeated_lines_df = pd.DataFrame(repeat_rows, columns=['source','filename','line','occurrences'])

print('DOCUMENT STATISTICS')
display(document_stats)
print('PAGE STATISTICS — first 20 rows')
display(page_stats.head(20))
print('REPEATED LINES (3+ occurrences) — possible headers/footers or repeated content')
display(repeated_lines_df.head(30))


DOCUMENT STATISTICS


,source,filename,file_size_KB,pages,total_words,total_characters,average_words_per_page,average_characters_per_page,empty_pages,low_text_pages,extractable_pages_percent,repeated_lines_3plus
0,Kaggle,kaggle.pdf,238.60,5,1256,8189,251.20,1637.80,0,0,100.0,0
1,College,clgdataset (3).pdf,333.27,27,2130,15221,78.89,563.74,0,0,100.0,15


PAGE STATISTICS — first 20 rows


,source,filename,page,words,characters,empty_page,low_text_page_under_50_chars
0,Kaggle,kaggle.pdf,1,24,169,False,False
1,Kaggle,kaggle.pdf,2,295,1828,False,False
2,Kaggle,kaggle.pdf,3,377,2371,False,False
3,Kaggle,kaggle.pdf,4,442,2925,False,False
4,Kaggle,kaggle.pdf,5,108,793,False,False
5,College,clgdataset (3).pdf,1,25,205,False,False
6,College,clgdataset (3).pdf,2,36,275,False,False
7,College,clgdataset (3).pdf,3,88,658,False,False
8,College,clgdataset (3).pdf,4,66,482,False,False
9,College,clgdataset (3).pdf,5,91,647,False,False


REPEATED LINES (3+ occurrences) — possible headers/footers or repeated content


,source,filename,line,occurrences
0,College,clgdataset (3).pdf,Definition and philosophy,28
1,College,clgdataset (3).pdf,Rational Agents,3
2,College,clgdataset (3).pdf,Nature of Environment,30
3,College,clgdataset (3).pdf,Thinking Humanly,3
4,College,clgdataset (3).pdf,Vaccum Cleaner Agent,3
5,College,clgdataset (3).pdf,Goal based Agents,4
6,College,clgdataset (3).pdf,Rational Agents QuantifyThinking IntelligenceH...,3
7,College,clgdataset (3).pdf,Artificial Intelligence,13
8,College,clgdataset (3).pdf,Rational Agents IdealVaccumRationalCleanerAgen...,10
9,College,clgdataset (3).pdf,Example,3


## 5. Inspect extracted text

Review each source's extracted text below. Compare it against the same pages in the original PDF. This is necessary to identify errors that basic statistics cannot detect.

In [24]:
for source, text in extracted_texts.items():
    print('\\n' + '=' * 80)
    print(f'SOURCE: {source}')
    print('=' * 80)
    print(text[:6000])
    if len(text) > 6000:
        print('\\n[Preview truncated; full text is saved in extraction_results/extracted_text/]')


\n================================================================================
SOURCE: Kaggle
===== PAGE 1 =====
Peer Review Framework for Predictive Analytics
in Humanitarian Response

MODEL REPORT:
Typhoon impact model
510 - An initiative of the Netherlands Red Cross

August 2022

===== PAGE 2 =====
Model Report:
Typhoon impact model


1. Background
This document summarises the documentation and the finding of the peer review of the trigger
used in OCHA’s Philippines Anticipatory Action framework for typhoons. The impact prediction
model used as the basis for the trigger was developed by the Netherlands Red Cross 510 data
and digital team, on behalf of the IFRC network and its partners. The model uses typhoon track
and rainfall forecasts to predict the percentage of houses that will be severely damaged per
municipality in the Philippines. OCHA Philippines used this model as a basis to begin a response
before a typhoon hit, based on the predicted number of houses damaged in the re

## 6. Manual quality review

For each PDF, inspect representative pages (for example, first, middle, and last pages, plus any low-text pages). Enter `Yes`, `No`, or `N/A` for each category. Add a short note describing a real example. Do not mark an issue unless you observed it in the original PDF versus extracted text.

In [27]:

# SECTION 6: Analysis of the Eight PDF Text Extraction Problems
# Produces written analysis, not a table.

import fitz
import re
from collections import Counter

def analyse_pdf_extraction(source, filename):
    doc = fitz.open(filename)
    page_texts = [
        page.get_text("text", sort=True).strip()
        for page in doc
    ]
    full_text = "\n".join(page_texts)

    print("\n" + "=" * 75)
    print(f"EXTRACTION QUALITY ANALYSIS: {source}")
    print("=" * 75)

    # 1. Missing text
    empty_pages = [
        i + 1 for i, text in enumerate(page_texts) if not text
    ]
    low_text_pages = [
        i + 1 for i, text in enumerate(page_texts)
        if 0 < len(text) < 50
    ]

    print("\n1. MISSING TEXT")
    if empty_pages:
        print(f"- No text was extracted from page(s): {empty_pages}.")
        print("- Check these pages in the original PDF to determine whether "
              "they contain images, scanned text, or genuinely no text.")
    else:
        print("- Text was extracted from every page.")
        if low_text_pages:
            print(f"- Page(s) with fewer than 50 extracted characters: "
                  f"{low_text_pages}. Check whether visible content is missing.")
        else:
            print("- No completely empty or very low-text pages were detected.")
        print("- Compare representative pages with the original to confirm "
              "that visible text has not been omitted.")

    # 2. Broken or garbled text
    replacement_chars = full_text.count("\ufffd")
    null_chars = full_text.count("\x00")
    suspicious_tokens = re.findall(
        r"\b[A-Za-z]{1,2}\s+[A-Za-z]{1,2}\s+[A-Za-z]{1,2}\b",
        full_text
    )

    print("\n2. BROKEN OR GARBLED TEXT")
    if replacement_chars or null_chars:
        print(f"- Warning signs detected: {replacement_chars} replacement "
              f"characters and {null_chars} null characters.")
    else:
        print("- No replacement or null characters were detected.")
    if suspicious_tokens:
        print(f"- Found {len(suspicious_tokens)} short, fragmented letter "
              "sequences that may deserve inspection.")
    else:
        print("- No obvious fragmented-letter pattern was detected.")
    print("- These checks cannot identify every garbled word; read the "
          "extracted text and compare questionable passages with the PDF.")

    # 3. Reading order
    print("\n3. READING ORDER")
    print("- Text was extracted in page order using PyMuPDF's sorted text output.")
    print("- Reading order cannot be confirmed from extracted text alone.")
    print("- Compare pages containing columns, multiple text boxes, bullet "
          "points, or diagrams to check whether sentences and paragraphs "
          "appear in the correct sequence.")

    # 4. Headers and footers
    normalized_lines = [
        re.sub(r"\s+", " ", line).strip()
        for text in page_texts
        for line in text.splitlines()
        if len(line.strip()) > 2
    ]
    line_counts = Counter(normalized_lines)
    repeated_lines = [
        (line, count) for line, count in line_counts.items()
        if count >= 3
    ]

    print("\n4. HEADERS AND FOOTERS")
    if repeated_lines:
        print("- Lines repeated at least three times across the document:")
        for line, count in sorted(
            repeated_lines, key=lambda item: item[1], reverse=True
        )[:8]:
            print(f"  • {line!r} — {count} occurrences")
        print("- Check whether these are page titles, slide headings, page "
              "numbers, or legitimate repeated content.")
    else:
        print("- No identical text lines occurring three or more times "
              "were detected.")
    print("- Repeated lines are possible headers/footers, not proof of an error.")

    # 5. Special symbols
    unusual_symbols = sorted(set(
        ch for ch in full_text
        if ord(ch) > 127 and not ch.isspace()
        and ch not in "“”‘’–—…"
    ))

    print("\n5. SPECIAL SYMBOLS")
    if unusual_symbols:
        print("- Non-ASCII symbols found include: "
              + " ".join(repr(ch) for ch in unusual_symbols[:30]))
        print("- Check symbols such as Greek letters, arrows, inequalities, "
              "and mathematical signs against the original PDF.")
    else:
        print("- No non-ASCII symbols were detected by this simple check.")
    print("- Ordinary-looking output does not guarantee that every symbol "
          "was extracted correctly.")

    # 6. Equations
    equation_hints = re.findall(
        r"[=<>≤≥±√∑∫∞∂∆πλμσαβγθ²³₀₁₂₃₄₅₆₇₈₉]",
        full_text
    )

    print("\n6. EQUATIONS")
    if equation_hints:
        print(f"- Found {len(equation_hints)} equation-related symbols or "
              "superscript/subscript characters.")
        print("- Inspect equations in the original PDF to check whether "
              "fractions, powers, subscripts, and symbol positions were preserved.")
    else:
        print("- No common equation-related symbols were detected.")
        print("- If the PDF contains equations, inspect them visually; equations "
              "may be missing or represented as images.")
    print("- Symbol detection alone does not measure equation accuracy.")

    # 7. Tables
    table_hints = re.findall(
        r"\t|(?:\S+\s{2,}\S+)",
        full_text
    )

    print("\n7. TABLES")
    if table_hints:
        print("- Spacing or tab patterns that may come from aligned content "
              "were detected.")
    else:
        print("- No obvious tab or multi-space alignment patterns were detected.")
    print("- Inspect any tables in the original PDF to check whether columns, "
          "rows, labels, and values remain correctly associated.")
    print("- A table may extract as plain text without preserving its structure.")

    # 8. Repeated content
    repeated_pages = []
    for i in range(1, len(page_texts)):
        previous = re.sub(r"\s+", " ", page_texts[i - 1]).strip()
        current = re.sub(r"\s+", " ", page_texts[i]).strip()
        if previous and current and previous == current:
            repeated_pages.append((i, i + 1))

    print("\n8. REPEATED CONTENT")
    if repeated_pages:
        print(f"- Identical consecutive page-text pairs detected: {repeated_pages}.")
        print("- Check whether these are duplicated pages or intentionally "
              "repeated material.")
    else:
        print("- No identical consecutive pages were detected.")
    if repeated_lines:
        print("- Repeated lines were also found; use the header/footer analysis "
              "above to distinguish normal repetition from possible duplication.")
    print("- This check detects exact repetition, not all near-duplicates.")

    print("\nOVERALL ASSESSMENT")
    print("- Automated checks identify possible warning signs, not definitive errors.")
    print("- Final conclusions should be based on comparing the extracted text "
          "with representative pages of the original PDF.")
    doc.close()


# Analyse both sources already mapped in SOURCE_FILES
for source, filename in SOURCE_FILES.items():
    analyse_pdf_extraction(source, filename)



EXTRACTION QUALITY ANALYSIS: Kaggle

1. MISSING TEXT
- Text was extracted from every page.
- No completely empty or very low-text pages were detected.
- Compare representative pages with the original to confirm that visible text has not been omitted.

2. BROKEN OR GARBLED TEXT
- No replacement or null characters were detected.
- Found 1 short, fragmented letter sequences that may deserve inspection.
- These checks cannot identify every garbled word; read the extracted text and compare questionable passages with the PDF.

3. READING ORDER
- Text was extracted in page order using PyMuPDF's sorted text output.
- Reading order cannot be confirmed from extracted text alone.
- Compare pages containing columns, multiple text boxes, bullet points, or diagrams to check whether sentences and paragraphs appear in the correct sequence.

4. HEADERS AND FOOTERS
- No identical text lines occurring three or more times were detected.
- Repeated lines are possible headers/footers, not proof of an error

## 7. Summarise and compare the two sources

Since there is only one PDF per source in this starter dataset, these are **document-level comparisons**, not general conclusions about every Kaggle or college PDF. A stronger study should include more PDFs per source.

In [26]:
numeric_columns = [
    'pages', 'total_words', 'total_characters', 'average_words_per_page',
    'average_characters_per_page', 'empty_pages', 'low_text_pages',
    'extractable_pages_percent', 'repeated_lines_3plus'
]
comparison = document_stats.set_index('source')[numeric_columns].T
display(comparison)

print('CAUTION: Higher word/character counts do not automatically mean better extraction.')
print('Check the original PDFs and manual quality-review notes before deciding which source is better.')


source,Kaggle,College
pages,5.0,27.00
total_words,1256.0,2130.00
total_characters,8189.0,15221.00
average_words_per_page,251.2,78.89
average_characters_per_page,1637.8,563.74
empty_pages,0.0,0.00
low_text_pages,0.0,0.00
extractable_pages_percent,100.0,100.00
repeated_lines_3plus,0.0,15.00


CAUTION: Higher word/character counts do not automatically mean better extraction.
Check the original PDFs and manual quality-review notes before deciding which source is better.


## 8. Export results and download them

This creates CSV files for document statistics, page statistics, repeated lines, and the quality-review template, plus a ZIP containing the extracted text files and reports.

In [25]:
import zipfile

os.makedirs('extraction_results', exist_ok=True)
document_stats.to_csv('extraction_results/document_statistics.csv', index=False)
page_stats.to_csv('extraction_results/page_statistics.csv', index=False)
repeated_lines_df.to_csv('extraction_results/repeated_lines.csv', index=False)
quality_review.to_csv('extraction_results/quality_review_template.csv', index=False)

zip_path = 'pdf_extraction_analysis_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk('extraction_results'):
        for name in filenames:
            full_path = os.path.join(root, name)
            zf.write(full_path, arcname=full_path)

print('Saved:', zip_path)
from google.colab import files
files.download(zip_path)


Saved: pdf_extraction_analysis_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Report-writing template

Use your actual outputs to fill in this summary. Do not claim one source is better until you have checked the extracted text manually.

**Objective:** This analysis compares PDF text extraction for the supplied Kaggle and college-material PDFs.

**Method:** Text was extracted using PyMuPDF. Document-level and page-level word/character counts were calculated. Pages with no text or fewer than 50 characters were flagged for review. Extracted text was compared manually with the source PDFs for missing/broken text, reading order, headers/footers, symbols, equations, tables, and repeated content.

**Results:** Insert actual page counts, word counts, character counts, and issue observations from the generated tables.

**Limitations:** This starter comparison contains only one PDF per source. Text count is not a direct measure of extraction accuracy; image-only pages may require OCR, and equations/tables require visual inspection.

**Conclusion:** State what the evidence supports, identify the document type(s) that caused errors, and recommend next steps.